#### Figure S1: Piecewise linear regression and SSA trend decomposition for sea ice extent timeseries

**Figure Caption**

Figure S1. Additional timeseries analysis of March sea ice extent. Top row shows piecewise linear regression with automatically-detected breakpoints for a) pan-Arctic and b) Greenland Sea regions. Bottom row shows Singular Spectrum Analysis (SSA) decomposition with identified maxima (purple triangles) and minima (teal inverted triangles) for c) pan-Arctic and d) Greenland Sea regions.

In [ ]:
"""
Figure S1: Piecewise Regression and SSA Decomposition
=====================================================
4-panel supplementary figure:
- Row 1 (a,b): Piecewise linear regression analysis
- Row 2 (c,d): SSA trend decomposition with peaks/troughs

Panel a: Pan-Arctic piecewise regression
Panel b: Greenland Sea piecewise regression
Panel c: Pan-Arctic SSA decomposition
Panel d: Greenland Sea SSA decomposition
"""

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import pwlf
from pyts.decomposition import SingularSpectrumAnalysis
from scipy.signal import find_peaks
from scipy import stats
import os
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

# File paths
ERA5_PATH = '../era5/era5_*_Arctic.nc'

# Analysis configuration
CONFIG = {
    'season': 'March',
    'n_segments': 2  # Number of piecewise segments
}

# Greenland Sea polygon
GREENLAND_SEA_COORDS = [(-22, 71), (-8.5, 71), (12, 79), (-21, 79), (-28, 73), (-22, 71)]

# Colors
COLORS = {
    'data': '#808080',
    'pwlr_fit': '#E63946',  # Red for piecewise fit
    'breakpoint': '#808080',
    'ssa_trend': '#1D3557',  # Navy for SSA trend
    'peaks': '#A23B72',  # Purple-red
    'troughs': '#2E8B57'  # Sea green
}

# ============================================================================
# FUNCTIONS
# ============================================================================

def analytical_trend_ci(x, y, confidence_level=0.95):
    """
    Calculate analytical confidence intervals for linear trend.
    
    Args:
        x: Independent variable (time)
        y: Dependent variable (SIE)
        confidence_level: Confidence level (default 0.95 for 95% CI)
    
    Returns:
        Dictionary with slope, intercept, and CI bounds
    """
    result = stats.linregress(x, y)
    
    n = len(x)
    y_pred = result.slope * x + result.intercept
    residuals = y - y_pred
    mse = np.sum(residuals**2) / (n - 2)
    
    x_mean = np.mean(x)
    sxx = np.sum((x - x_mean)**2)
    se_slope = np.sqrt(mse / sxx)
    se_intercept = np.sqrt(mse * (1/n + x_mean**2/sxx))
    
    from scipy.stats import t
    t_val = t.ppf((1 + confidence_level) / 2, n - 2)
    
    return {
        'slope': result.slope,
        'intercept': result.intercept,
        'slope_ci_lower': result.slope - t_val * se_slope,
        'slope_ci_upper': result.slope + t_val * se_slope,
        'intercept_ci_lower': result.intercept - t_val * se_intercept,
        'intercept_ci_upper': result.intercept + t_val * se_intercept,
        'r_squared': result.rvalue**2,
        'p_value': result.pvalue
    }

def perform_ssa_analysis(data, times, window_fraction=0.15):
    """
    Perform SSA trend decomposition.
    
    Args:
        data: Time series data
        times: Time coordinates (for reference)
        window_fraction: Fraction of data length for window size
    
    Returns:
        Dictionary with trend, peaks, and troughs
    """
    data = np.asarray(data)
    
    n = len(data)
    window_size = max(10, int(window_fraction * n))
    
    ssa = SingularSpectrumAnalysis(window_size=window_size, groups=None)
    X_ssa = ssa.fit_transform(data.reshape(1, -1))
    
    # Trend is first two components
    trend = X_ssa[0, 0] + X_ssa[0, 1]
    
    # Find peaks and troughs
    peaks, _ = find_peaks(trend, distance=int(0.05 * n))
    troughs, _ = find_peaks(-trend, distance=int(0.05 * n))
    
    return {
        'trend': trend,
        'peaks': peaks,
        'troughs': troughs
    }

# ============================================================================
# LOAD DATA
# ============================================================================

print("="*70)
print("LOADING DATA")
print("="*70)

# Load ERA5 sea ice concentration
print("\nLoading ERA5 sea ice data...")
ds = xr.open_mfdataset(ERA5_PATH, chunks={'time': 12})
sic = ds['siconc']
print(f"  Loaded: {sic.dims}")
print(f"  Time range: {str(sic.time.min().values)[:10]} to {str(sic.time.max().values)[:10]}")

# Load NSIDC sea ice index data
print("\nLoading NSIDC sea ice index data...")
url_nh = 'https://noaadata.apps.nsidc.org/NOAA/G02135/north/daily/data/N_seaice_extent_daily_v3.0.csv'
df_nh = pd.read_csv(url_nh, skiprows=2)
df_nh.columns = ['year', 'month', 'day', 'extent', 'missing', 'source_data']
df_nh['time'] = pd.to_datetime(df_nh[['year', 'month', 'day']])
ds_nh = df_nh.set_index('time')['extent'].to_xarray()
ds_nh = ds_nh.where(ds_nh > 0, drop=True)
ds_nh['month'] = ds_nh.time.dt.month

# Calculate Greenland Sea extent
print("\nCalculating Greenland Sea extent...")
from shapely.geometry import Point, Polygon
gs_polygon = Polygon(GREENLAND_SEA_COORDS)

lons_2d, lats_2d = np.meshgrid(sic.longitude.values, sic.latitude.values)
gs_mask = np.zeros_like(lons_2d, dtype=bool)
for i in range(lons_2d.shape[0]):
    for j in range(lons_2d.shape[1]):
        point = Point(lons_2d[i,j], lats_2d[i,j])
        gs_mask[i,j] = gs_polygon.contains(point)

cell_areas = (111.32 * 0.25) * (111.32 * 0.25 * np.cos(np.radians(lats_2d)))
sic_gs = sic.where(gs_mask)
ice_area_gs = (sic_gs * cell_areas).sum(dim=['latitude', 'longitude']) / 1e6
ds_gs = ice_area_gs.compute()
ds_gs['month'] = ds_gs.time.dt.month
print(f"  Greenland Sea extent calculated")

# ============================================================================
# CREATE FIGURE
# ============================================================================

print("\n" + "="*70)
print("CREATING FIGURE")
print("="*70)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

season_month = 3
season_name = CONFIG['season']

# ============================================================================
# ROW 1: PIECEWISE LINEAR REGRESSION
# ============================================================================

print("\n" + "="*70)
print("ROW 1: PIECEWISE LINEAR REGRESSION")
print("="*70)

# --- Panel (a): Pan-Arctic ---
ax = axes[0]
print(f"\nPanel (a): Pan-Arctic {season_name} piecewise regression")

x_data = ds_nh.time.where(ds_nh.month == season_month, drop=True)
y_data = ds_nh.where(ds_nh.month == season_month, drop=True)

# Convert to numeric for pwlf
x_num = x_data.dt.year.values + (x_data.dt.dayofyear.values / 365.25)
y_num = y_data.values

# Fit piecewise linear model
my_pwlf = pwlf.PiecewiseLinFit(x_num, y_num)
breaks = my_pwlf.fit(CONFIG['n_segments'])
y_hat = my_pwlf.predict(x_num)

# Get breakpoint as datetime
np_break = pd.Timestamp(year=int(breaks[1]), month=season_month, day=15)
np_break_date = pd.Timestamp(year=int(breaks[1]), month=1, day=1)

# Calculate statistics for each segment
seg1_mask = x_num <= breaks[1]
seg2_mask = x_num >= breaks[1]
seg1_stats = analytical_trend_ci(x_num[seg1_mask], y_data.values[seg1_mask])
seg2_stats = analytical_trend_ci(x_num[seg2_mask], y_data.values[seg2_mask])

print(f"Automated breakpoint: {str(np_break_date)[0:4]}")
print(f"Segment 1 R²: {seg1_stats['r_squared']:.3f}, p: {seg1_stats['p_value']:.4f}")
print(f"Segment 2 R²: {seg2_stats['r_squared']:.3f}, p: {seg2_stats['p_value']:.4f}")

# Plot
ax.plot(x_data, y_data, color='#808080', linewidth=1, alpha=0.5, zorder=1)
ax.scatter(x_data, y_data, s=16, color=COLORS['data'], alpha=0.6,
           label='Observed', zorder=2, edgecolors='none')
ax.plot(x_data, y_hat, color=COLORS['pwlr_fit'], linewidth=2.5,
        label='Piecewise fit', zorder=3)
ax.axvline(x=np_break_date, color=COLORS['breakpoint'], linestyle=':',
           linewidth=1.5, alpha=0.7,
           label=f"Breakpoint: {str(np_break_date)[0:4]}", zorder=1)

ax.set_xlabel('Year', fontsize=10)
ax.set_ylabel(r'Sea ice extent ($\times10^{6}$ km$^{2}$)', fontsize=10)
ax.set_title(f'a) Pan-Arctic, piecewise linear regression', loc='left',
            fontsize=11, fontweight='bold')
ax.legend(loc='upper right', frameon=True, edgecolor='none', fontsize=9)
ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)

# --- Panel (b): Greenland Sea ---
ax = axes[1]
print(f"\nPanel (b): Greenland Sea {season_name} piecewise regression")

x_data = ds_gs.time.where(ds_gs.month == season_month, drop=True)
y_data = ds_gs.where(ds_gs.month == season_month, drop=True)

# Convert to numeric
x_num = x_data.dt.year.values + (x_data.dt.dayofyear.values / 365.25)
y_num = y_data.values

# Fit piecewise linear model
my_pwlf = pwlf.PiecewiseLinFit(x_num, y_num)
breaks = my_pwlf.fit(CONFIG['n_segments'])
y_hat = my_pwlf.predict(x_num)

np_break = pd.Timestamp(year=int(breaks[1]), month=season_month, day=15)
np_break_date = pd.Timestamp(year=int(breaks[1]), month=1, day=1)

# Calculate statistics
seg1_mask = x_num <= breaks[1]
seg2_mask = x_num >= breaks[1]
seg1_stats = analytical_trend_ci(x_num[seg1_mask], y_data.values[seg1_mask])
seg2_stats = analytical_trend_ci(x_num[seg2_mask], y_data.values[seg2_mask])

print(f"Automated breakpoint: {str(np_break_date)[0:4]}")
print(f"Segment 1 R²: {seg1_stats['r_squared']:.3f}, p: {seg1_stats['p_value']:.4f}")
print(f"Segment 2 R²: {seg2_stats['r_squared']:.3f}, p: {seg2_stats['p_value']:.4f}")

# Plot
ax.plot(x_data, y_data, color='#808080', linewidth=1, alpha=0.5, zorder=1)
ax.scatter(x_data, y_data, s=16, color=COLORS['data'], alpha=0.6,
           label='Observed', zorder=2, edgecolors='none')
ax.plot(x_data, y_hat, color=COLORS['pwlr_fit'], linewidth=2.5,
        label='Piecewise fit', zorder=3)
ax.axvline(x=np_break_date, color=COLORS['breakpoint'], linestyle=':',
           linewidth=1.5, alpha=0.7,
           label=f"Breakpoint: {str(np_break_date)[0:4]}", zorder=1)

ax.set_xlabel('Year', fontsize=10)
ax.set_title(f'b) Greenland Sea, piecewise linear regression', loc='left',
            fontsize=11, fontweight='bold')
ax.legend(loc='upper right', frameon=True, edgecolor='none', fontsize=9)
ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)

# ============================================================================
# ROW 2: SSA TREND DECOMPOSITION
# ============================================================================

print("\n" + "="*70)
print("ROW 2: SSA TREND DECOMPOSITION")
print("="*70)

# --- Panel (c): Pan-Arctic ---
ax = axes[2]
print(f"\nPanel (c): Pan-Arctic {season_name} SSA")

x_data = ds_nh.time.where(ds_nh.month == season_month, drop=True)
y_data = ds_nh.where(ds_nh.month == season_month, drop=True)

ssa_results = perform_ssa_analysis(y_data.values, x_data)

# Plot observed data
ax.plot(x_data, y_data, color='#808080', linewidth=1, alpha=0.5, zorder=1)
ax.scatter(x_data, y_data, s=16, color='#808080', alpha=0.6,
           label='Observed', zorder=2, edgecolors='none')

# Plot SSA trend
ax.plot(x_data, ssa_results['trend'], color=COLORS['ssa_trend'],
        linewidth=2, label='SSA trend', alpha=0.8, zorder=3)

# Mark peaks and troughs
peaks = ssa_results['peaks']
troughs = ssa_results['troughs']

ax.scatter(x_data[peaks], ssa_results['trend'][peaks],
          marker='^', s=80, color=COLORS['peaks'],
          label='Maxima', zorder=5, edgecolor='white', linewidth=0.5)
ax.scatter(x_data[troughs], ssa_results['trend'][troughs],
          marker='v', s=80, color=COLORS['troughs'],
          label='Minima', zorder=5, edgecolor='white', linewidth=0.5)

# Annotate peaks
for i in peaks:
    ax.annotate(f"{str(x_data[i].data)[0:4]}",
               (x_data[i], ssa_results['trend'][i]),
               textcoords="offset points", xytext=(0, 10),
               ha='center', fontsize=7, color=COLORS['peaks'],
               fontweight='bold')

# Annotate troughs
for i in troughs:
    ax.annotate(f"{str(x_data[i].data)[0:4]}",
               (x_data[i], ssa_results['trend'][i]),
               textcoords="offset points", xytext=(0, -15),
               ha='center', fontsize=7, color=COLORS['troughs'],
               fontweight='bold')

print(f"Number of peaks detected: {len(peaks)}")
print(f"Number of troughs detected: {len(troughs)}")

ax.set_xlabel('Year', fontsize=10)
ax.set_ylabel(r'Sea ice extent ($\times10^{6}$ km$^{2}$)', fontsize=10)
ax.set_title(f'c) Pan-Arctic, SSA decomposition', loc='left',
            fontsize=11, fontweight='bold')
ax.legend(loc='upper right', frameon=True, edgecolor='none', ncol=2, fontsize=9)
ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)

# --- Panel (d): Greenland Sea ---
ax = axes[3]
print(f"\nPanel (d): Greenland Sea {season_name} SSA")

x_data = ds_gs.time.where(ds_gs.month == season_month, drop=True)
y_data = ds_gs.where(ds_gs.month == season_month, drop=True)

ssa_results = perform_ssa_analysis(y_data.values, x_data)

# Plot observed data
ax.plot(x_data, y_data, color='#808080', linewidth=1, alpha=0.5, zorder=1)
ax.scatter(x_data, y_data, s=16, color='#808080', alpha=0.6,
           label='Observed', zorder=2, edgecolors='none')

# Plot SSA trend
ax.plot(x_data, ssa_results['trend'], color=COLORS['ssa_trend'],
        linewidth=2, label='SSA trend', alpha=0.8, zorder=3)

# Mark peaks and troughs
peaks = ssa_results['peaks']
troughs = ssa_results['troughs']

ax.scatter(x_data[peaks], ssa_results['trend'][peaks],
          marker='^', s=80, color=COLORS['peaks'],
          label='Maxima', zorder=5, edgecolor='white', linewidth=0.5)
ax.scatter(x_data[troughs], ssa_results['trend'][troughs],
          marker='v', s=80, color=COLORS['troughs'],
          label='Minima', zorder=5, edgecolor='white', linewidth=0.5)

# Annotate peaks
for i in peaks:
    ax.annotate(f"{str(x_data[i].data)[0:4]}",
               (x_data[i], ssa_results['trend'][i]),
               textcoords="offset points", xytext=(0, 10),
               ha='center', fontsize=7, color=COLORS['peaks'],
               fontweight='bold')

# Annotate troughs
for i in troughs:
    ax.annotate(f"{str(x_data[i].data)[0:4]}",
               (x_data[i], ssa_results['trend'][i]),
               textcoords="offset points", xytext=(0, -15),
               ha='center', fontsize=7, color=COLORS['troughs'],
               fontweight='bold')

print(f"Number of peaks detected: {len(peaks)}")
print(f"Number of troughs detected: {len(troughs)}")

ax.set_xlabel('Year', fontsize=10)
ax.set_title(f'd) Greenland Sea, SSA decomposition', loc='left',
            fontsize=11, fontweight='bold')
ax.legend(loc='upper right', frameon=True, edgecolor='none', ncol=2, fontsize=9)
ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)

# ============================================================================
# SAVE FIGURE
# ============================================================================

plt.tight_layout()

output_dir = './figures'
os.makedirs(output_dir, exist_ok=True)

output_path = f"{output_dir}/Figure_S1_piecewise_and_SSA.png"
fig.savefig(output_path, dpi=600, bbox_inches='tight', facecolor='white')
print(f"\n{'='*70}")
print(f"Figure saved to: {output_path}")
print(f"{'='*70}\n")

plt.show()

print("\nConfiguration used:")
print(f"  Season: {CONFIG['season']}")
print(f"  Piecewise segments: {CONFIG['n_segments']}")